# Descriptive Analytics — End-to-End Pipeline

This notebook runs the **ten steps of descriptive analytics** as a single, reproducible pipeline on a **synthetic retail dataset** generated in code (no external files required).

It is the companion to the *Data Analytics: Tackling the Data* series at mzamudio.com. Each section maps to one step's article and uses the same pandas patterns shown there.

**Pipeline:** Collect → Clean → Transform & Aggregate → Filter → Segment → Visualize → Compare → Report → Patterns → Share.

**Requirements:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`.

## Step 1 — Collecting Data

In production, collection extracts and unions data from databases, APIs, and files. Here we **generate a synthetic retail orders dataset** instead, so the pipeline is fully reproducible. We deliberately inject duplicates, inconsistent text, nulls, and a few bad rows so the cleaning step has real work to do.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N = 20_000

# Skewed customer base so a minority of customers drive most orders (Pareto)
n_cust = 2_000
ranks = np.arange(1, n_cust + 1)
weights = 1 / ranks
weights = weights / weights.sum()

order_date = pd.to_datetime('2022-01-01') + pd.to_timedelta(rng.integers(0, 730, N), unit='D')

# Seasonal demand: Q4 lifts volume, Jan-Feb is quieter
month_num = order_date.month
season = np.select([month_num.isin([11, 12]), month_num.isin([1, 2])],
                   [1.8, 0.7], default=1.0)
quantity = np.maximum(1, (rng.integers(1, 12, N) * season).round().astype(int))

orders = pd.DataFrame({
    'order_id': np.arange(1, N + 1),
    'order_date': order_date,
    'region': rng.choice(['West', 'East', 'Central', 'South'], N),
    'category': rng.choice(['Furniture', 'Office', 'Technology'], N),
    'segment': rng.choice(['Consumer', 'Corporate', 'Home Office'], N),
    'customer_id': rng.choice(ranks, size=N, p=weights),
    'quantity': quantity,
    'discount': rng.choice([0.0, 0.1, 0.2, 0.3], N, p=[0.5, 0.25, 0.15, 0.1]),
})

# A handful of bulk B2B orders -> genuine high-value outliers for the filter step
bulk = pd.DataFrame({
    'order_id': np.arange(N + 1, N + 31),
    'order_date': pd.to_datetime('2023-06-15'),
    'region': 'Central', 'category': 'Technology', 'segment': 'Corporate',
    'customer_id': rng.integers(1, n_cust, 30),
    'quantity': rng.integers(60, 150, 30),
    'discount': 0.0,
})
orders = pd.concat([orders, bulk], ignore_index=True)

# Inject realistic messiness for the cleaning step: duplicates,
# inconsistent text casing, missing discounts, and zero-quantity rows.
dupes = orders.sample(300, random_state=1)
orders = pd.concat([orders, dupes], ignore_index=True)
orders.loc[orders.sample(200, random_state=2).index, 'region'] = ' west '
orders.loc[orders.sample(150, random_state=3).index, 'discount'] = np.nan
orders.loc[orders.sample(50, random_state=4).index, 'quantity'] = 0

print(f'Generated {len(orders):,} raw order rows')
orders.head()

## Step 2 — Cleaning Data

Remove duplicates, coerce data types, standardize categorical text, fill missing discounts, and drop invalid rows (no date, zero quantity).

In [ ]:
# 1. Remove duplicate orders
orders = orders.drop_duplicates(subset='order_id')

# 2. Fix data types
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
orders['quantity'] = pd.to_numeric(orders['quantity'], errors='coerce')

# 3. Standardize categorical text
for col in ['region', 'category', 'segment']:
    orders[col] = orders[col].str.strip().str.title()

# 4. Handle missing values and drop invalid rows
orders['discount'] = orders['discount'].fillna(0)
orders = orders.dropna(subset=['order_id', 'order_date'])
orders = orders[orders['quantity'] > 0]

print(f'{len(orders):,} clean rows')
print(orders.isna().sum())

## Step 3 — Transforming & Aggregation

Engineer the `sales` and `profit` features from quantity, price, and discount, extract the order month, then aggregate to monthly metrics by region.

In [ ]:
# Feature engineering: unit price by category, then sales and profit
price = {'Furniture': 180, 'Office': 25, 'Technology': 320}
orders['unit_price'] = orders['category'].map(price)
orders['sales'] = orders['quantity'] * orders['unit_price'] * (1 - orders['discount'])
orders['profit'] = orders['sales'] * (0.30 - orders['discount'])
orders['month'] = orders['order_date'].dt.to_period('M').dt.to_timestamp()

# Aggregate to monthly metrics by region
monthly = (orders.groupby(['month', 'region'])
                 .agg(sales=('sales', 'sum'),
                      profit=('profit', 'sum'),
                      orders=('order_id', 'nunique'))
                 .reset_index())
monthly.head()

## Step 4 — Filtering & Reducing Noise

Scope the data to real sales, then trim extreme sales outliers with the IQR rule so they do not distort averages and trends.

In [ ]:
# Relevance filter: real sales only
focused = orders[orders['sales'] > 0].copy()

# Noise reduction: trim sales outliers with the IQR rule
q1, q3 = focused['sales'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
focused = focused[focused['sales'].between(low, high)]

print(f'{len(orders):,} rows -> {len(focused):,} after outlier trim')
print(f'Sales bounds kept: {low:,.0f} to {high:,.0f}')

## Step 5 — Segmentation & Clustering

Build a customer-level table, assign rule-based spend tiers with `qcut`, and discover natural cohorts with K-Means on scaled features.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Customer-level table
cust = (focused.groupby('customer_id')
               .agg(total_sales=('sales', 'sum'),
                    orders=('order_id', 'nunique'),
                    avg_discount=('discount', 'mean'))
               .reset_index())

# Rule-based spend tiers
cust['tier'] = pd.qcut(cust['total_sales'], q=3, labels=['Low', 'Mid', 'High'])

# K-Means clustering on scaled features
X = StandardScaler().fit_transform(cust[['total_sales', 'orders', 'avg_discount']])
cust['cluster'] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X)

cust.groupby('cluster')[['total_sales', 'orders', 'avg_discount']].mean()

## Step 6 — Visualization & Trending

Plot monthly sales with a 3-month rolling average to expose the underlying trend and seasonality.

In [ ]:
# Monthly total sales with a 3-month rolling trend
series = focused.groupby('month')['sales'].sum()
trend = series.rolling(window=3, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 4))
series.plot(ax=ax, alpha=0.5, label='Monthly sales')
trend.plot(ax=ax, linewidth=2, label='3-month trend')
ax.set_title('Monthly sales with rolling trend')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
plt.show()

## Step 7 — Comparing

Compare performance year-over-year, and across categories and regions, to see what is growing and what is lagging.

In [ ]:
# Year-over-year comparison of total sales
yearly = focused.assign(year=focused['order_date'].dt.year).groupby('year')['sales'].sum()
yoy = yearly.pct_change() * 100
print(yearly)
print('YoY % change:')
print(yoy.round(1))

# Category x region comparison
pivot = focused.pivot_table(index='category', columns='region', values='sales', aggfunc='sum')
pivot.round(0)

## Step 8 — Reporting

Consolidate the analysis into headline KPIs and a summary table by region and category — the core of a performance report.

In [ ]:
# Headline KPIs
kpis = {
    'total_sales': focused['sales'].sum(),
    'total_profit': focused['profit'].sum(),
    'overall_margin': focused['profit'].sum() / focused['sales'].sum(),
}
print({k: round(v, 3) for k, v in kpis.items()})

# Summary table by region and category
summary = (focused.groupby(['region', 'category'])
                  .agg(sales=('sales', 'sum'), profit=('profit', 'sum'))
                  .assign(margin=lambda d: (d['profit'] / d['sales']).round(3))
                  .sort_values('sales', ascending=False)
                  .reset_index())
summary

## Step 9 — Patterns & Insights

Interpret the results: correlations with profit, a Pareto concentration of customers, and anomalous months. Patterns are hypotheses — correlation is not causation.

In [ ]:
# 1. Correlation of numeric drivers with profit
corr = focused[['sales', 'quantity', 'discount', 'profit']].corr()
print('Correlation with profit:')
print(corr['profit'].sort_values())

# 2. Pareto concentration of customers
cust_sales = focused.groupby('customer_id')['sales'].sum().sort_values(ascending=False)
cum_share = cust_sales.cumsum() / cust_sales.sum()
n_top = max(1, int(0.20 * len(cust_sales)))
print(f'Top 20% of customers = {cum_share.iloc[n_top - 1]:.0%} of sales')

# 3. Anomalous months (>2 std dev from the mean)
m = focused.groupby('month')['sales'].sum()
z = (m - m.mean()) / m.std()
print('Anomalous months:')
print(m[z.abs() > 2])

## Step 10 — Sharing & Publishing

Export the summary to CSV and a styled HTML table from a single source of truth, ready for a portal, email, or dashboard.

In [ ]:
# Publish the summary from a single source of truth
summary.to_csv('retail_summary.csv', index=False)

try:
    styled = (summary.style
                     .format({'sales': '${:,.0f}', 'profit': '${:,.0f}'})
                     .background_gradient(subset=['sales'], cmap='Blues'))
    styled.to_html('retail_summary.html')
    out = styled
except (ImportError, AttributeError):
    # .style needs the optional jinja2 dependency; fall back to a plain HTML table
    summary.to_html('retail_summary.html', index=False)
    out = summary

print('Exported retail_summary.csv and retail_summary.html')
out

## Wrap-Up

Starting from raw, messy, synthetic data, the pipeline produced a cleaned and enriched dataset, customer segments, a sales trend, year-over-year comparisons, a KPI report, interpretable insights, and published deliverables — the full arc of descriptive analytics in one reproducible flow.

Back to the series: **Data Analytics: Tackling the Data** at mzamudio.com.